In [ ]:
import pandas as pd

df = pd.read_csv(r'C:\Users\DELL\Downloads\fetal_health.csv')

print(df.shape)
print(df.head())
print(df.info())
print(df.isnull().sum())
print(df["fetal_health"].value_counts())
X = df.drop("fetal_health", axis=1)
y = df["fetal_health"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef
)
results = []
for name, model in models.items():
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_test, y_pred)

    # AUC for multi-class
    try:
        auc = roc_auc_score(y_test, model.predict_proba(X_test), multi_class='ovr')
    except:
        auc = 0

    results.append([name, acc, auc, prec, rec, f1, mcc])
    

results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"
])

print(results_df)

import os
import joblib

# Create models folder if not exists
os.makedirs("../models", exist_ok=True)

# Save models into models folder
for name, model in models.items():
    filename = f"../models/{name}.pkl"
    joblib.dump(model, filename)

# Save scaler
joblib.dump(scaler, "../models/scaler.pkl")

test_df = pd.DataFrame(X_test, columns=X.columns)
test_df["fetal_health"] = y_test.values
test_df.to_csv("../test_data.csv", index=False)